
# News Feed System Design (HLD + LLD)

## Problem Statement

Design a scalable News Feed system like:
- Facebook Feed
- Instagram Feed
- LinkedIn Feed
- Twitter/X Timeline

Users should:
- create posts
- follow users
- see personalized feeds
- like/comment/share posts

---

# Functional Requirements

- User can create posts
- Follow/unfollow users
- Personalized feed generation
- Feed pagination
- Like/comment/share support
- Real-time updates

---

# Non Functional Requirements

- High availability
- Low latency
- Scalability
- Fault tolerance
- High read throughput

---

# High Level Architecture (HLD)

```text
                 +----------------+
                 |     Clients    |
                 +--------+-------+
                          |
                          v
                 +----------------+
                 | Load Balancer  |
                 +--------+-------+
                          |
        +----------------+----------------+
        |                                 |
        v                                 v
+----------------+              +----------------+
| Feed Service   |              | Post Service   |
+--------+-------+              +--------+-------+
         |                               |
         v                               v
+----------------+              +----------------+
| Redis Cache    |              | Media Storage  |
+--------+-------+              +----------------+
         |
         v
+----------------+
| Feed Generator |
+--------+-------+
         |
         v
+----------------+
| Databases      |
+----------------+
```

---

# Core Components

| Component | Purpose |
|---|---|
| Post Service | Create/manage posts |
| Feed Service | Generate timelines |
| Feed Generator | Push/pull feed logic |
| Cache Layer | Fast feed retrieval |
| Graph Service | Followers/following |
| Ranking Service | Personalized ranking |

---

# Feed Generation Approaches

## Push Model (Fan-out on Write)

When user posts:
- push post to followers' feeds

### Pros
- fast reads

### Cons
- expensive writes
- hard for celebrity users

---

## Pull Model (Fan-out on Read)

Generate feed during read request.

### Pros
- cheaper writes

### Cons
- slower reads

---

## Hybrid Model (Recommended)

Most systems use:
- Push for normal users
- Pull for celebrity users

---

# Post Creation Flow

```text
1. User creates post
2. Store post in DB
3. Send event to Kafka
4. Feed generator processes event
5. Push feed updates
6. Cache updated feeds
```

---

# Feed Read Flow

```text
1. User opens app
2. Feed service checks Redis cache
3. Return cached feed
4. Missing items fetched from DB
5. Paginated response returned
```

---

# Database Design

## User Table

| Column | Type |
|---|---|
| user_id | bigint |
| name | varchar |

---

## Post Table

| Column | Type |
|---|---|
| post_id | bigint |
| user_id | bigint |
| content | text |
| media_url | varchar |
| created_at | timestamp |

---

## Follow Table

| follower_id | followee_id |
|---|---|

---

## Feed Table

| user_id | post_id | timestamp |
|---|---|---|

---

# Redis Cache

Cache:
- user feeds
- hot posts
- follower lists

Example:

```text
feed:user123
```

---

# Pagination

Use:
- cursor-based pagination

Example:

```http
GET /feed?cursor=abc123
```

Better than offset pagination.

---

# Ranking System

Feed ranking factors:
- likes
- comments
- shares
- recency
- relationship score
- engagement probability

---

# Real-Time Updates

Need:
- WebSockets
- Push notifications

---

# Media Storage

Images/videos stored separately.

Use:
- S3
- CDN

Posts only store metadata/URLs.

---

# CDN Integration

Benefits:
- edge caching
- low latency
- reduced backend load

---

# Scaling Strategy

## Horizontal Scaling

Scale:
- feed servers
- cache nodes
- DB shards

---

## Sharding

Shard by:

```text
hash(user_id) % N
```

---

# Hot User Problem

Celebrity users:
- millions of followers

Push model becomes expensive.

### Solution
- pull model for celebrities
- async queues
- batch processing

---

# Kafka/Event Queue

Used for:
- feed updates
- notifications
- analytics

Flow:

```text
Post Created
    ↓
Kafka
    ↓
Feed Workers
```

---

# Failure Handling

## Cache Failure
Fallback to DB.

## Queue Failure
Use retries and dead letter queues.

## DB Failure
Use replication and failover.

---

# Consistency

Feeds are usually:
- eventually consistent

Small delays acceptable.

---

# Monitoring

Track:
- feed latency
- cache hit ratio
- Kafka lag
- DB load

Tools:
- Prometheus
- Grafana

---

# Security

Need:
- authentication
- authorization
- spam detection
- rate limiting

---

# Low Level Design (LLD)

Main Classes:

```text
UserService
PostService
FeedService
FeedGenerator
RankingService
CacheService
NotificationService
```

---

# User Entity

```java
class User {
    Long userId;
    String name;
}
```

---

# Post Entity

```java
class Post {

    Long postId;
    Long userId;
    String content;
    String mediaUrl;
    Date createdAt;
}
```

---

# Feed Item

```java
class FeedItem {

    Long userId;
    Long postId;
    Date timestamp;
}
```

---

# Feed Service

```java
class FeedService {

    public List<Post> getFeed(Long userId) {

        List<Post> posts = cache.get(userId);

        if(posts != null) {
            return posts;
        }

        posts = repository.fetchFeed(userId);

        cache.put(userId, posts);

        return posts;
    }
}
```

---

# Feed Generator

```java
class FeedGenerator {

    public void fanOut(Post post) {

        List<Long> followers =
            graphService.getFollowers(post.userId);

        for(Long follower : followers) {

            feedRepository.insert(follower, post.postId);
        }
    }
}
```

---

# Sequence Diagram

## Post Creation

```text
Client
  ↓
Post Service
  ↓
Database
  ↓
Kafka
  ↓
Feed Generator
  ↓
Follower Feeds Updated
```

---

## Feed Read

```text
Client
  ↓
Feed Service
  ↓
Redis Cache
  ↓ (miss)
Database
  ↓
Return Feed
```

---

# Bottlenecks

| Bottleneck | Solution |
|---|---|
| Celebrity fanout | Hybrid push/pull |
| DB overload | Redis cache |
| Feed ranking latency | Precompute rankings |
| Media bandwidth | CDN |
| Queue lag | Partitioned Kafka |

---

# Final Production Architecture

```text
                  +----------------+
                  |    Clients     |
                  +--------+-------+
                           |
                           v
                  +----------------+
                  | Load Balancer  |
                  +--------+-------+
                           |
        +------------------+------------------+
        |                                     |
        v                                     v
+----------------+                  +----------------+
| Feed Service   |                  | Post Service   |
+--------+-------+                  +--------+-------+
         |                                     |
         v                                     v
+----------------+                  +----------------+
| Redis Cache    |                  | Media Storage  |
+--------+-------+                  +----------------+
         |
         v
+----------------+
| Kafka Queue    |
+--------+-------+
         |
         v
+----------------+
| Feed Generator |
+--------+-------+
         |
         v
+----------------+
| Distributed DB |
+----------------+
```

---

# Final Interview Summary

A scalable news feed system can be designed using:
- Hybrid push/pull feed generation
- Redis caching for fast reads
- Kafka for asynchronous fanout
- Distributed databases for scalability
- ML ranking for personalization
- CDN for media delivery

The system is highly read-heavy, so feed caching and scalable fanout are critical.
